In [ ]:
!pip install kagglehub -q

In [ ]:
import kagglehub
path = kagglehub.dataset_download("ichhadhari/indian-birds")
print("Path to dataset files:", path)

In [ ]:
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'models').exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / 'models').exists() and (parent / 'notebooks').exists():
            PROJECT_ROOT = parent
            break

MODELS_DIR = PROJECT_ROOT / 'models' / 'vision'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_PATH = MODELS_DIR / 'best_model.pth'
FINAL_MODEL_PATH = MODELS_DIR / 'indian_birds_image_classifier.pth'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'Model directory: {MODELS_DIR}')


In [ ]:
# Find the folder that contains actual class subfolders
def find_image_root(base_path):
    for root, dirs, files in os.walk(base_path):
        imgs = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if imgs:
            return os.path.dirname(root)
    return base_path

data_root = find_image_root(path)
print("Data root:", data_root)
print("Classes found:", os.listdir(data_root)[:5], "...")

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 15
LR = 1e-4

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [ ]:
full_dataset = datasets.ImageFolder(data_root)
num_classes = len(full_dataset.classes)
print(f"Total images: {len(full_dataset)}")
print(f"Number of classes: {num_classes}")

val_size = int(0.15 * len(full_dataset))
test_size = int(0.10 * len(full_dataset))
train_size = len(full_dataset) - val_size - test_size

train_ds, val_ds, test_ds = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

# Apply transforms
train_ds.dataset.transform = train_tfms

val_loader_ds = datasets.ImageFolder(data_root, transform=val_tfms)
_, val_ds_clean, test_ds_clean = random_split(
    val_loader_ds, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds_clean, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds_clean, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train: {train_size} | Val: {val_size} | Test: {test_size}")

In [ ]:
# Quick look at some samples
classes = full_dataset.classes
imgs, labels = next(iter(train_loader))

mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flatten()):
    img = imgs[i] * std + mean
    ax.imshow(img.permute(1,2,0).clamp(0,1))
    ax.set_title(classes[labels[i]], fontsize=8)
    ax.axis('off')
plt.suptitle('Sample Training Images', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# EfficientNet-B0: lightweight but punches above its weight for image classification
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# Freeze early layers, fine-tune the last few blocks
for name, param in model.named_parameters():
    param.requires_grad = False

for name, param in model.named_parameters():
    if 'features.6' in name or 'features.7' in name or 'features.8' in name or 'classifier' in name:
        param.requires_grad = True

# Replace classifier head
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features, 512),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(512, num_classes)
)

model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}")

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.set_grad_enabled(train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            loss = criterion(out, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * imgs.size(0)
            correct    += (out.argmax(1) == labels).sum().item()
            total      += imgs.size(0)

    return total_loss / total, correct / total


history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer, train=True)
    vl_loss, vl_acc = run_epoch(model, val_loader,   criterion, train=False)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    flag = ''
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        flag = '  ← saved'

    print(f"Epoch {epoch:02d}/{EPOCHS}  "
          f"Train Loss: {tr_loss:.4f}  Acc: {tr_acc:.4f}  |  "
          f"Val Loss: {vl_loss:.4f}  Acc: {vl_acc:.4f}{flag}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'],   label='Val')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.legend()

ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'],   label='Val')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.legend()

plt.tight_layout()
plt.show()
print(f"Best Val Accuracy: {best_val_acc:.4f}")

In [ ]:
# Load best weights and evaluate on test set
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        preds = model(imgs).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

test_acc = (all_preds == all_labels).mean()
print(f"Test Accuracy: {test_acc:.4f}")
print()
print(classification_report(all_labels, all_preds, target_names=classes))

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(max(10, num_classes), max(8, num_classes - 2)))
sns.heatmap(cm, annot=(num_classes <= 20), fmt='d',
            xticklabels=classes, yticklabels=classes,
            cmap='Blues', linewidths=0.5)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — Test Set')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
# Show a few test predictions
model.eval()
sample_imgs, sample_labels = next(iter(test_loader))
with torch.no_grad():
    sample_preds = model(sample_imgs.to(device)).argmax(1).cpu()

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flatten()):
    img = sample_imgs[i] * std + mean
    ax.imshow(img.permute(1,2,0).clamp(0,1))
    pred  = classes[sample_preds[i]]
    truth = classes[sample_labels[i]]
    color = 'green' if pred == truth else 'red'
    ax.set_title(f"P: {pred}\nT: {truth}", color=color, fontsize=7)
    ax.axis('off')
plt.suptitle('Predictions (Green=Correct, Red=Wrong)', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Save final model with class mapping
torch.save({
    'model_state_dict': model.state_dict(),
    'classes': classes,
    'num_classes': num_classes,
    'best_val_acc': best_val_acc,
    'test_acc': test_acc
}, FINAL_MODEL_PATH)

print(f"Model saved to: {FINAL_MODEL_PATH}")
print(f"Val Acc:  {best_val_acc:.4f}")
print(f"Test Acc: {test_acc:.4f}")